In [7]:
import requests
import time

base_url = "https://rest.uniprot.org/uniprotkb/search"
query = "organism_id:9606 AND reviewed:true"
fields = "accession,gene_names,protein_name,cc_subcellular_location"
batch_size = 500
format_type = "tsv"

# Output files
unfiltered_file = "all_human_reviewed.tsv"
filtered_file = "strict_cell_membrane_proteins.tsv"

# Inclusion and exclusion rules
include_terms = {"plasma membrane", "cell membrane"}
exclude_terms = {
    "mitochondria", "lysosome", "endoplasmic", "golgi", "nucleus",
    "cytoplasm", "peroxisome", "ribosome", "nucleolus", "centrosome",
    "cytosol", "vacuole", "nuclear membrane", "extracellular"
}

# Initial request with cursor-based pagination
params = {
    "query": query,
    "fields": fields,
    "format": format_type,
    "size": batch_size
}

print("Downloading all reviewed human proteins from UniProt...")

# Store all results
all_lines = []
cursor = None
page_count = 0

while True:
    if cursor:
        params["cursor"] = cursor
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    lines = response.text.strip().split("\n")

    if page_count == 0:
        all_lines.extend(lines)  # include header
    else:
        all_lines.extend(lines[1:])  # skip duplicate headers

    page_count += 1
    print(f"Downloaded page {page_count} with {len(lines)-1} entries.")

    # Get next cursor
    next_cursor = response.links.get("next", {}).get("url")
    if not next_cursor:
        break
    # Extract cursor from the URL
    cursor = next_cursor.split("cursor=")[-1].split("&")[0]
    time.sleep(1)  # Be gentle to the server

# Save unfiltered file
with open(unfiltered_file, "w", encoding="utf-8") as f:
    f.write("\n".join(all_lines))
print(f"✅ Saved all reviewed human proteins to '{unfiltered_file}'")

# Filter for strict plasma/cell membrane localization
header = all_lines[0]
filtered_lines = [header]

for line in all_lines[1:]:
    loc = line.lower()
    if any(term in loc for term in include_terms) and not any(term in loc for term in exclude_terms):
        filtered_lines.append(line)

# Save filtered file
with open(filtered_file, "w", encoding="utf-8") as f:
    f.write("\n".join(filtered_lines))
print(f"✅ Saved {len(filtered_lines)-1} strict membrane proteins to '{filtered_file}'")


Downloaded page 1 with 500 entries.
Downloaded page 2 with 500 entries.
Downloaded page 3 with 500 entries.
Downloaded page 4 with 500 entries.
Downloaded page 5 with 500 entries.
Downloaded page 6 with 500 entries.
Downloaded page 7 with 500 entries.
Downloaded page 8 with 500 entries.
Downloaded page 9 with 500 entries.
Downloaded page 10 with 500 entries.
Downloaded page 11 with 500 entries.
Downloaded page 12 with 500 entries.
Downloaded page 13 with 500 entries.
Downloaded page 14 with 500 entries.
Downloaded page 15 with 500 entries.
Downloaded page 16 with 500 entries.
Downloaded page 17 with 500 entries.
Downloaded page 18 with 500 entries.
Downloaded page 19 with 500 entries.
Downloaded page 20 with 500 entries.
Downloaded page 21 with 500 entries.
Downloaded page 22 with 500 entries.
Downloaded page 23 with 500 entries.
Downloaded page 24 with 500 entries.
Downloaded page 25 with 500 entries.
Downloaded page 26 with 500 entries.
Downloaded page 27 with 500 entries.
Downloaded